In [1]:
import cellxgene_census
import pandas as pd
import scanpy as sc
import numpy as np
import json
import anndata as ad
import re
import obonet
import networkx as nx
from rapidfuzz import process

In [2]:
cell_gene_dir = "../../data/censusxgene"
# cell_types = [
#     "malignant cell",
#     "luminal epithelial cell of mammary gland",
#     "basal-myoepithelial cell of mammary gland",
#     "fibroblast of mammary gland",
#     "macrophage",
#     "T cell",
#     "B cell"
# ]

tissues = ['breast', 'lung', 'kidney', 'bladder organ'] # 15M cells
tissues_general = ['breast', 'lung', 'kidney', 'bladder organ'] # 17.9M cells
# simplified & only cell types that are in all tissues - # 12M cells
# only cell types that are in all tissueas - # 5M cells
 
with open("../../data/protein_coding_genes.txt", "r") as f:
    protein_coding_genes = [line.strip() for line in f]

In [3]:
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    obs_df = cellxgene_census.get_obs(
        census,
        "homo_sapiens",
        value_filter=f"tissue_general in {tissues_general} and suspension_type == 'cell'" # and cell_type in {cell_types}"
    )

In [4]:
obs_df

,soma_joinid,dataset_id,assay,assay_ontology_term_id,cell_type,cell_type_ontology_term_id,development_stage,development_stage_ontology_term_id,disease,disease_ontology_term_id,...,tissue,tissue_ontology_term_id,tissue_type,tissue_general,tissue_general_ontology_term_id,raw_sum,nnz,raw_mean_nnz,raw_variance_nnz,n_measured_vars
0,31188,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,Schwann cell,CL:0002573,18th week post-fertilization stage,HsapDv:0000055,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,5780.0,2236,2.584973,42.338638,16035
1,31189,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,Schwann cell,CL:0002573,15th week post-fertilization stage,HsapDv:0000052,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,2087.0,1357,1.537951,5.754643,16035
2,31190,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,Schwann cell,CL:0002573,15th week post-fertilization stage,HsapDv:0000052,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,14733.0,4268,3.451968,107.457969,16035
3,31191,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,immature Schwann cell,CL:0002377,Carnegie stage 19,HsapDv:0000026,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,5174.0,1797,2.879243,53.412470,16035
4,31192,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,neuron,CL:0000540,Carnegie stage 19,HsapDv:0000026,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,8107.0,3033,2.672931,31.431249,16035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17890872,156173185,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,"CD8-positive, alpha-beta T cell",CL:0000625,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,4561.0,2016,2.262401,53.842278,60606
17890873,156173186,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,capillary endothelial cell,CL:0002144,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,8823.0,3366,2.621212,68.019033,60606
17890874,156173187,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,endothelial cell of artery,CL:1000413,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,12442.0,3943,3.155465,133.749290,60606
17890875,156173188,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,macrophage,CL:0000235,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,38498.0,6121,6.289495,994.178270,60606


In [5]:
counts = (
    obs_df
    .groupby(["cell_type"], observed=True) #"tissue_general", "tissue",
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

counts.to_csv(f"{cell_gene_dir}/cell_counts_per_group_path.csv", index=False)

cell_types = set(obs_df['cell_type'].values)
len(cell_types)

360

## CEll type names generalizing

In [6]:
def simplify_cell_type(name):
    """
    Funkcja mapująca skomplikowane nazwy komórek na ich główne linie biologiczne.
    Kolejność instrukcji 'if' ma znaczenie (od najbardziej do najmniej specyficznych).
    """
    if pd.isna(name):
        return "Unknown"
        
    name = str(name).lower()
    
    # 1. Limfocyty T (T cells) - zachowujemy podział na główne subpopulacje
    if "regulatory t cell" in name or "t-regulatory" in name: return "Regulatory T cell (Treg)"
    if "cd4-positive" in name or "cd4+" in name or "t-helper" in name or "helper t cell" in name: return "CD4+ T cell"
    if "cd8-positive" in name or "cd8+" in name or "tc1" in name or "cytotoxic t cell" in name: return "CD8+ T cell"
    if "gamma-delta t cell" in name: return "Gamma-delta T cell"
    if "nk t cell" in name: return "NK T cell"
    if "thymocyte" in name: return "Thymocyte"
    if "t cell" in name or "lymphocyte" in name: return "T cell" # Reszta T / ogólne limfocyty
    
    # 2. Limfocyty B (B cells)
    if "plasma cell" in name or "plasmablast" in name: return "Plasma cell"
    if "b cell" in name or "pre-b" in name or "pro-b" in name: return "B cell"
    
    # 3. Komórki NK i wrodzone komórki limfoidalne (ILC)
    if "natural killer" in name or "nk cell" in name or "nkp46" in name: return "Natural killer cell"
    if "innate lymphoid" in name: return "Innate lymphoid cell (ILC)"
    
    # 4. Linia mieloidalna (Makrofagi, Monocyty, Dendrytyczne)
    if "macrophage" in name or "kupffer" in name: return "Macrophage"
    if "monocyte" in name or "promonocyte" in name: return "Monocyte"
    if "plasmacytoid dendritic" in name: return "Plasmacytoid dendritic cell"
    if "dendritic cell" in name or "langerhans" in name: return "Dendritic cell"
    if "neutrophil" in name: return "Neutrophil"
    if "basophil" in name: return "Basophil"
    if "eosinophil" in name: return "Eosinophil"
    if "mast cell" in name: return "Mast cell"
    if "megakaryocyte" in name or "platelet" in name: return "Megakaryocyte/Platelet"
    if "erythro" in name or "red blood cell" in name or "reticulocyte" in name: return "Erythroid cell"
    if "myeloid" in name or "granulocyte" in name: return "Myeloid cell" # Ogólne mieloidalne
    
    # 5. Komórki zrębu (Stromal) i wspierające
    if "fibroblast" in name or "fibrocyte" in name: return "Fibroblast"
    if "pericyte" in name or "mural cell" in name: return "Pericyte"
    # if "smooth muscle" in name: return "Smooth muscle cell"
    # if "cardiac muscle" in name: return "Cardiac muscle cell"
    # if "skeletal muscle" in name: return "Skeletal muscle cell"
    if "muscle" in name: return "Muscle cell"
    if "endothelial" in name or "lymphangioblast" in name or "vasa recta" in name: return "Endothelial cell"
    if "chondrocyte" in name: return "Chondrocyte"
    if "stromal cell" in name or "mesenchymal" in name: return "Mesenchymal/Stromal cell"
    if "mesothelial" in name: return "Mesothelial cell"
    
    # 6. Komórki nabłonkowe (Epithelial) i ich specyficzne typy
    if "basal cell" in name or "basal-myoepithelial" in name or "myoepithelial" in name or "suprabasal" in name: return "Basal/Myoepithelial cell"
    if "goblet" in name: return "Goblet cell"
    if "ciliated" in name: return "Ciliated cell"
    if "club cell" in name: return "Club cell"
    if "neuroendocrine" in name: return "Neuroendocrine cell"
    if "ionocyte" in name: return "Ionocyte"
    if "tuft cell" in name: return "Tuft cell"
    if "acinar" in name: return "Acinar cell"
    if "podocyte" in name: return "Podocyte"
    if "mesangial" in name: return "Mesangial cell"
    if "hepatocyte" in name: return "Hepatocyte"
    if "enterocyte" in name: return "Enterocyte"
    if "keratinocyte" in name: return "Keratinocyte"
    if "urothelial" in name: return "Urothelial cell"
    if "alveolar type 1" in name or "alveolar type 2" in name or "alveolar type i" in name or "alveolar epithelial" in name: return "Alveolar epithelial cell"
    if "epithelial" in name or "luminal" in name or "tubule" in name or "duct" in name or "secretory" in name or "lactocyte" in name or "secreting" in name: return "Epithelial cell"
    
    # 7. Układ nerwowy
    if "neuron" in name or "neural" in name: return "Neuron"
    if "schwann" in name or "glial" in name: return "Glial cell"
    
    # 8. Komórki macierzyste/progenitorowe
    if "stem cell" in name: return "Stem cell"
    if "progenitor" in name: return "Progenitor cell"
    
    # 9. Inne
    if "malignant" in name or "abnormal" in name: return "Malignant/Abnormal cell"
    if "unknown" in name: return "Unknown"
    
    # Fallback (jeśli jakaś komórka nie wpadła w żadną kategorię, 
    # zostaw oryginał z dużą literą na początku)
    return str(name).capitalize()



In [7]:
# --- APLIKACJA FUNKCJI NA TWOIM DATAFRAME ---

# 1. Tworzymy nową kolumnę z uproszczonymi nazwami (na wypadek gdybyś chciał zachować oryginał)
obs_df["cell_type_broad"] = obs_df["cell_type"].apply(simplify_cell_type)

# Rzutujemy na kategorię (opcjonalne, ale przyspiesza działanie groupby)
obs_df["cell_type_broad"] = obs_df["cell_type_broad"].astype("category")

# 2. Generowanie nowych zliczeń
counts_broad = (
    obs_df
    .groupby(["cell_type_broad"], observed=False) 
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

# 3. Zapis do pliku
counts_broad.to_csv(f"{cell_gene_dir}/cell_counts_per_group_path_simplified.csv", index=False)

In [8]:
# <1000 or unknown wylatuja

# 1. Obliczamy liczebność każdego typu komórki
counts = obs_df["cell_type_broad"].value_counts()

# 2. Wybieramy nazwy typów komórek, które mają co najmniej 1000 rekordów
#    i jednocześnie NIE są nazwane "unknown"
valid_cell_types = counts[
    (counts >= 1000) & (counts.index.str.lower() != "unknown")
].index

# 3. Filtrujemy oryginalny DataFrame, zostawiając tylko poprawne rekordy
obs_df_filtered = obs_df[obs_df["cell_type_broad"].isin(valid_cell_types)].copy()

# 4. Jeśli cell_type był typu 'category', warto usunąć nieużywane już kategorie (jak unknown)
if obs_df_filtered["cell_type_broad"].dtype.name == "category":
    obs_df_filtered["cell_type_broad"] = obs_df_filtered["cell_type_broad"].cat.remove_unused_categories()

print(f"Liczba rekordów przed filtrowaniem: {len(obs_df)}")
print(f"Liczba rekordów po filtrowaniu:    {len(obs_df_filtered)}")
print(f"Liczba usuniętych typów komórek:   {len(counts) - len(valid_cell_types)}")

Liczba rekordów przed filtrowaniem: 17890877
Liczba rekordów po filtrowaniu:    16534664
Liczba usuniętych typów komórek:   27


In [9]:
np.random.seed(42)

cells_per_type = 100000
selected_ids = []

for cell_type in valid_cell_types:
    subset_ct = obs_df[obs_df["cell_type_broad"] == cell_type]
    n_available = len(subset_ct)
    n_sample = min(cells_per_type, n_available)
    
    ids = np.random.choice(subset_ct["soma_joinid"].values, n_sample, replace=False)
    selected_ids.extend(ids)

    print(cell_type, n_available, n_sample)

Epithelial cell 2691358 100000
Macrophage 2131076 100000
Fibroblast 1684334 100000
Endothelial cell 1584354 100000
T cell 1496266 100000
Alveolar epithelial cell 882940 100000
CD4+ T cell 806693 100000
CD8+ T cell 792797 100000
Basal/Myoepithelial cell 679676 100000
Monocyte 590686 100000
Natural killer cell 500917 100000
B cell 402709 100000
Perivascular cell 367507 100000
Ciliated cell 254693 100000
Dendritic cell 240229 100000
Plasma cell 173490 100000
Regulatory T cell (Treg) 162196 100000
Muscle cell 153623 100000
Pericyte 125175 100000
Urothelial cell 81906 81906
Mesenchymal/Stromal cell 74310 74310
Myeloid cell 73821 73821
Neutrophil 65091 65091
Mesangial cell 55497 55497
Stem cell 53218 53218
Neuroendocrine cell 48598 48598
NK T cell 46634 46634
Mononuclear phagocyte 44342 44342
Malignant/Abnormal cell 38169 38169
Innate lymphoid cell (ILC) 32586 32586
Leukocyte 31319 31319
Plasmacytoid dendritic cell 26326 26326
T follicular helper cell 24518 24518
Kidney interstitial cell 185

In [10]:
obs_df_filtered = obs_df[obs_df["soma_joinid"].isin(selected_ids)]

In [11]:
cell_types = set(obs_df_filtered['cell_type_broad'].values)

len(cell_types), cell_types

(55,
 {'Acinar cell',
  'Adventitial cell',
  'Alveolar epithelial cell',
  'B cell',
  'Basal/Myoepithelial cell',
  'Basophil',
  'CD4+ T cell',
  'CD8+ T cell',
  'Chondrocyte',
  'Ciliated cell',
  'Dendritic cell',
  'Endothelial cell',
  'Epithelial cell',
  'Erythroid cell',
  'Fibroblast',
  'Gamma-delta T cell',
  'Glial cell',
  'Hematopoietic cell',
  'Hepatocyte',
  'Innate lymphoid cell (ILC)',
  'Ionocyte',
  'Kidney cell',
  'Kidney interstitial cell',
  'Leukocyte',
  'Macrophage',
  'Malignant/Abnormal cell',
  'Megakaryocyte/Platelet',
  'Mesangial cell',
  'Mesenchymal/Stromal cell',
  'Mesothelial cell',
  'Monocyte',
  'Mononuclear phagocyte',
  'Muscle cell',
  'Myeloid cell',
  'NK T cell',
  'Natural killer cell',
  'Neuroendocrine cell',
  'Neutrophil',
  'Pericyte',
  'Perivascular cell',
  'Plasma cell',
  'Plasmacytoid dendritic cell',
  'Podocyte',
  'Progenitor cell',
  'Regulatory T cell (Treg)',
  'Renal alpha-intercalated cell',
  'Renal beta-intercalat

In [12]:
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    cell_adata = cellxgene_census.get_anndata(
        census,
        "homo_sapiens",
        obs_coords=selected_ids,
        column_names=["assay", "cell_type", "donor_id", "tissue", "tissue_general"]
    )

/tmp/ipykernel_2081656/795482341.py:2: FutureWarning: The argument `column_names` is deprecated and will be removed in a future release. Please use `obs_column_names` and `var_column_names` instead.
  cell_adata = cellxgene_census.get_anndata(
/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [13]:
cols_you_want = [
    "soma_joinid",
    "cell_type",
    "donor_id",
    "tissue",
    "tissue_general"
]

cell_adata.obs = cell_adata.obs[cols_you_want]
cell_adata.obs["cell_type"] = cell_adata.obs["cell_type"].cat.remove_unused_categories()

In [14]:
cell_adata.obs["cell_type"] = cell_adata.obs["cell_type"].apply(simplify_cell_type)

In [15]:
cell_adata.obs['cell_type'].value_counts()[:25]

cell_type
B cell                      100000
Plasma cell                 100000
T cell                      100000
Fibroblast                  100000
Endothelial cell            100000
Macrophage                  100000
Ciliated cell               100000
Natural killer cell         100000
Dendritic cell              100000
Muscle cell                 100000
CD4+ T cell                 100000
Epithelial cell             100000
Pericyte                    100000
Monocyte                    100000
CD8+ T cell                 100000
Alveolar epithelial cell    100000
Perivascular cell           100000
Basal/Myoepithelial cell    100000
Regulatory T cell (Treg)    100000
Urothelial cell              81906
Mesenchymal/Stromal cell     74310
Myeloid cell                 73821
Neutrophil                   65091
Mesangial cell               55497
Stem cell                    53218
Name: count, dtype: int64

In [16]:
protein_set = set(protein_coding_genes)

mask = cell_adata.var["feature_name"].apply(lambda x: x in protein_set)
cell_adata = cell_adata[:, mask].copy()

In [17]:
sc.pp.normalize_total(cell_adata, target_sum=1e4) #, inplace=False)
sc.pp.log1p(cell_adata)

In [18]:
print(cell_adata)
print("obs columns:", cell_adata.obs.columns)
print("var shape:", cell_adata.var.shape)
print("X type:", type(cell_adata.X))

AnnData object with n_obs × n_vars = 2713945 × 19930
    obs: 'soma_joinid', 'cell_type', 'donor_id', 'tissue', 'tissue_general'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'
    uns: 'log1p'
obs columns: Index(['soma_joinid', 'cell_type', 'donor_id', 'tissue', 'tissue_general'], dtype='object')
var shape: (19930, 7)
X type: <class 'scipy.sparse._csr.csr_matrix'>


In [19]:
vocab_path = '/scratch/2370352/my-research/papers/scgpt/save/whole_human/vocab.json'

with open(vocab_path, "r") as f:
    vocab = json.load(f)

model_genes = list(vocab.keys())

print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(model_genes[:10])  # przykładowe pierwsze 10 genów

Liczba genów w scGPT vocab: 60697
['RP5-973N23.5', 'RP11-182N22.10', 'CTB-53D8.3', 'RP11-348N17.2', 'RP11-205M20.8', 'RP11-326C3.17', 'RP11-439H13.3', 'RP11-413H22.3', 'GET1-SH3BGR', 'CH17-476P10.1']


In [20]:
# adata_genes = lista genów z adata
# model_genes = lista genów z scGPT vocab

adata_genes = cell_adata.var['feature_name'].tolist()

# zamień na sety dla szybkiego porównania
adata_set = set(adata_genes)
model_set = set(model_genes)

# wspólne geny
common_genes = adata_set & model_set

# geny w adata, których nie ma w scGPT
missing_in_model = adata_set - model_set

# geny w scGPT, których nie ma w adata
missing_in_adata = model_set - adata_set

print(f"Liczba genów w adata: {len(adata_genes)}")
print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(f"Liczba genów wspólnych: {len(common_genes)}")
print(f"Liczba genów w adata nie w vocab: {len(missing_in_model)}")
print(f"Liczba genów w vocab nie w adata: {len(missing_in_adata)}")

# przykładowe geny
print("Przykłady genów wspólnych:", list(common_genes)[:10])
print("Przykłady genów w adata ale nie w vocab:", list(missing_in_model)[:10])
print("Przykłady genów w vocab ale nie w adata:", list(missing_in_adata)[:10])

Liczba genów w adata: 19930
Liczba genów w scGPT vocab: 60697
Liczba genów wspólnych: 19193
Liczba genów w adata nie w vocab: 686
Liczba genów w vocab nie w adata: 41504
Przykłady genów wspólnych: ['FAM182B', 'CNBP', 'CD247', 'MIER1', 'SEMA4D', 'CDIP1', 'RANBP10', 'OVOL3', 'G6PC1', 'RRN3']
Przykłady genów w adata ale nie w vocab: ['ENSG00000262526', 'ENSG00000284512', 'ENSG00000285602', 'ENSG00000267127', 'ENSG00000233539', 'ENSG00000268926', 'ENSG00000258529', 'ENSG00000249016', 'ENSG00000176349', 'ENSG00000267110']
Przykłady genów w vocab ale nie w adata: ['E2F6P4', 'RP11-452B18.2', 'RP5-872K7.7', 'MIR3664', 'LINC01358', 'RPL22P1', 'RN7SL635P', 'RP11-354G16.1', 'RP3-406M12.1', 'RPL35AP31']


In [21]:
gene_info = pd.read_csv("/scratch/2370352/my-research/data/gene_info_table.csv")  # lub pełna ścieżka

# Stwórz słownik: ensembl_id -> gene_name
ensg_to_symbol = dict(zip(gene_info['ensembl_id'], gene_info['gene_name']))

print(list(ensg_to_symbol.items())[:10])

[('ENSG00000000003', 'TSPAN6'), ('ENSG00000000005', 'TNMD'), ('ENSG00000000419', 'DPM1'), ('ENSG00000000457', 'SCYL3'), ('ENSG00000000460', 'C1orf112'), ('ENSG00000000938', 'FGR'), ('ENSG00000000971', 'CFH'), ('ENSG00000001036', 'FUCA2'), ('ENSG00000001084', 'GCLC'), ('ENSG00000001167', 'NFYA')]


In [22]:
# jeśli w adata masz geny zapisane jako ENSG, mapujemy je
mapped_genes = []

for g in cell_adata.var['feature_name']:
    if g in ensg_to_symbol:
        mapped_genes.append(ensg_to_symbol[g])
    else:
        mapped_genes.append(g)  # zachowaj tak jak jest, np. już symboliczny gen

# podmieniamy w adata.var
cell_adata.var['feature_name_mapped'] = mapped_genes

In [23]:
# adata_genes = lista genów z adata
# model_genes = lista genów z scGPT vocab

adata_genes = cell_adata.var['feature_name_mapped'].tolist()

# zamień na sety dla szybkiego porównania
adata_set = set(adata_genes)
model_set = set(model_genes)

# wspólne geny
common_genes = adata_set & model_set

# geny w adata, których nie ma w scGPT
missing_in_model = adata_set - model_set

# geny w scGPT, których nie ma w adata
missing_in_adata = model_set - adata_set

print(f"Liczba genów w adata: {len(adata_genes)}")
print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(f"Liczba genów wspólnych: {len(common_genes)}")
print(f"Liczba genów w adata nie w vocab: {len(missing_in_model)}")
print(f"Liczba genów w vocab nie w adata: {len(missing_in_adata)}")

# przykładowe geny
print("Przykłady genów wspólnych:", list(common_genes)[:10])
print("Przykłady genów w adata ale nie w vocab:", list(missing_in_model)[:10])
print("Przykłady genów w vocab ale nie w adata:", list(missing_in_adata)[:10])

Liczba genów w adata: 19930
Liczba genów w scGPT vocab: 60697
Liczba genów wspólnych: 19512
Liczba genów w adata nie w vocab: 309
Liczba genów w vocab nie w adata: 41185
Przykłady genów wspólnych: ['FAM182B', 'CNBP', 'CD247', 'MIER1', 'SEMA4D', 'CDIP1', 'RANBP10', 'OVOL3', 'G6PC1', 'RRN3']
Przykłady genów w adata ale nie w vocab: ['AC097625.2', 'AL157935.2', 'AC007204.1', 'AC091167.8', 'AC097636.2', 'AC006449.5', 'AC020915.6', 'AL162377.3', 'AC018523.2', 'FP565260.4']
Przykłady genów w vocab ale nie w adata: ['E2F6P4', 'RP11-452B18.2', 'RP5-872K7.7', 'MIR3664', 'LINC01358', 'RPL22P1', 'RN7SL635P', 'RP11-354G16.1', 'RP3-406M12.1', 'RPL35AP31']


In [24]:
np.random.seed(42)

all_indices = np.arange(cell_adata.n_obs)
np.random.shuffle(all_indices)

split = int(0.9 * len(all_indices))

train_indices = all_indices[:split]
test_indices  = all_indices[split:]

train_adata = cell_adata[train_indices].copy()
test_adata  = cell_adata[test_indices].copy()

In [25]:
train_adata.write("/scratch/2370352/my-research/adapter_premium/data_new/blkb_simp_100k_path_train.h5ad")
test_adata.write("/scratch/2370352/my-research/adapter_premium/data_new/blkb_simp_100k_path_test.h5ad")


In [26]:
adata = ad.read_h5ad("/scratch/2370352/my-research/adapter_premium/data_new/blkb_simp_100k_path_test.h5ad")


In [27]:
adata.obs['cell_type'].value_counts()

cell_type
Perivascular cell                10073
Endothelial cell                 10055
CD4+ T cell                      10047
Plasma cell                      10047
Macrophage                       10039
Regulatory T cell (Treg)         10037
Muscle cell                      10029
T cell                           10019
Ciliated cell                    10000
Monocyte                          9999
Natural killer cell               9956
Pericyte                          9949
Dendritic cell                    9940
Epithelial cell                   9933
Alveolar epithelial cell          9925
Basal/Myoepithelial cell          9917
Fibroblast                        9916
CD8+ T cell                       9896
B cell                            9871
Urothelial cell                   8264
Myeloid cell                      7497
Mesenchymal/Stromal cell          7425
Neutrophil                        6411
Mesangial cell                    5578
Stem cell                         5318
Neuroendocrine 